In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [2]:
import sys
import tensorflow as tf
from tensorflow.keras import layers, models
import warnings

import matplotlib.pyplot as plt

import numpy as np
import cv2

In [3]:
sys.path.insert(0, "../src/")
warnings.filterwarnings("ignore")

In [4]:
from image_preprocessing import load_dataset

In [5]:
DATASET_PATH = '../dataset/Garbage Classification Dataset/Garbage classification/Garbage classification'

In [6]:
import tensorflow as tf
import os
import random

def load_dataset(dataset_path):

    # -------- Collect all image paths --------
    image_paths = []
    labels = []

    class_names = sorted(os.listdir(dataset_path))

    for label_idx, class_name in enumerate(class_names):
        class_dir = os.path.join(dataset_path, class_name)

        if not os.path.isdir(class_dir):
            continue

        for file in os.listdir(class_dir):
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                image_paths.append(os.path.join(class_dir, file))
                labels.append(label_idx)

    # -------- Shuffle --------
    combined = list(zip(image_paths, labels))
    random.shuffle(combined)
    image_paths, labels = zip(*combined)

    total = len(image_paths)

    # -------- Split --------
    train_end = int(0.8 * total)
    val_end   = int(0.9 * total)

    train_paths = image_paths[:train_end]
    val_paths   = image_paths[train_end:val_end]
    test_paths  = image_paths[val_end:]

    train_labels = labels[:train_end]
    val_labels   = labels[train_end:val_end]
    test_labels  = labels[val_end:]

    # -------- Helper function --------
    def parse_image(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, (128, 128))
        return img, label

    # -------- Create datasets --------
    train_ds = tf.data.Dataset.from_tensor_slices((list(train_paths), list(train_labels)))
    val_ds   = tf.data.Dataset.from_tensor_slices((list(val_paths), list(val_labels)))
    test_ds  = tf.data.Dataset.from_tensor_slices((list(test_paths), list(test_labels)))

    # -------- Map + optimize --------
    AUTOTUNE = tf.data.AUTOTUNE

    train_ds = train_ds.map(parse_image, num_parallel_calls=AUTOTUNE)
    val_ds   = val_ds.map(parse_image, num_parallel_calls=AUTOTUNE)
    test_ds  = test_ds.map(parse_image, num_parallel_calls=AUTOTUNE)

    train_ds = train_ds.shuffle(1000).batch(32).prefetch(AUTOTUNE)
    val_ds   = val_ds.batch(32).prefetch(AUTOTUNE)
    test_ds  = test_ds.batch(32).prefetch(AUTOTUNE)

    return train_ds, val_ds, test_ds, class_names

In [7]:
train, test, val, classes = load_dataset(DATASET_PATH)

2026-04-28 16:13:40.921923: E tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:268] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [8]:
def create_model(num_classes):

    model = models.Sequential([

        layers.Input(shape=(128, 128, 3)),

        layers.Rescaling(1./255),
        layers.Conv2D(16, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),


        layers.Conv2D(32, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(64, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(96, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.GlobalAveragePooling2D(),

        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),

        layers.Dense(num_classes, activation='softmax')
    ])

    return model

In [9]:
model = create_model(len(classes))
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 rescaling (Rescaling)       (None, 128, 128, 3)       0         
                                                                 
 conv2d (Conv2D)             (None, 128, 128, 16)      448       
                                                                 
 batch_normalization (Batch  (None, 128, 128, 16)      64        
 Normalization)                                                  
                                                                 
 max_pooling2d (MaxPooling2  (None, 64, 64, 16)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 64, 64, 32)        4640      
                                                                 
 batch_normalization_1 (Bat  (None, 64, 64, 32)        1

In [11]:
import os
import tensorflow as tf

EPOCHS = 200

os.makedirs('models', exist_ok=True)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train,
    validation_data=val,
    epochs=EPOCHS,
    verbose=1)

Epoch 1/200
64/64 [==============================] - 20s 280ms/step - loss: 1.2672 - accuracy: 0.5383 - val_loss: 1.8045 - val_accuracy: 0.2253
Epoch 2/200
64/64 [==============================] - 21s 322ms/step - loss: 1.0881 - accuracy: 0.5903 - val_loss: 2.0089 - val_accuracy: 0.1858
Epoch 3/200
64/64 [==============================] - 18s 278ms/step - loss: 0.9989 - accuracy: 0.6487 - val_loss: 1.8664 - val_accuracy: 0.2411
Epoch 4/200
64/64 [==============================] - 23s 352ms/step - loss: 0.9200 - accuracy: 0.6734 - val_loss: 1.8647 - val_accuracy: 0.2688
Epoch 5/200
64/64 [==============================] - 21s 328ms/step - loss: 0.8766 - accuracy: 0.6893 - val_loss: 1.6096 - val_accuracy: 0.3676
Epoch 6/200
64/64 [==============================] - 21s 323ms/step - loss: 0.8337 - accuracy: 0.7056 - val_loss: 1.6458 - val_accuracy: 0.3794
Epoch 7/200
64/64 [==============================] - 21s 322ms/step - loss: 0.7965 - accuracy: 0.7110 - val_loss: 1.6227 - val_accuracy:

64/64 [==============================] - 21s 318ms/step - loss: 0.0701 - accuracy: 0.9777 - val_loss: 0.9656 - val_accuracy: 0.7352
Epoch 58/200
64/64 [==============================] - 20s 312ms/step - loss: 0.0882 - accuracy: 0.9758 - val_loss: 3.3628 - val_accuracy: 0.4941
Epoch 59/200
64/64 [==============================] - 21s 321ms/step - loss: 0.0510 - accuracy: 0.9852 - val_loss: 0.8148 - val_accuracy: 0.7787
Epoch 60/200
64/64 [==============================] - 21s 326ms/step - loss: 0.0959 - accuracy: 0.9723 - val_loss: 2.1817 - val_accuracy: 0.5810
Epoch 61/200
64/64 [==============================] - 21s 316ms/step - loss: 0.1074 - accuracy: 0.9649 - val_loss: 0.9208 - val_accuracy: 0.7549
Epoch 62/200
64/64 [==============================] - 21s 328ms/step - loss: 0.0837 - accuracy: 0.9723 - val_loss: 5.1506 - val_accuracy: 0.4150
Epoch 63/200
64/64 [==============================] - 20s 315ms/step - loss: 0.0968 - accuracy: 0.9693 - val_loss: 0.7132 - val_accuracy: 0.766

64/64 [==============================] - 30s 471ms/step - loss: 0.0197 - accuracy: 0.9921 - val_loss: 1.3694 - val_accuracy: 0.7115
Epoch 114/200
64/64 [==============================] - 25s 387ms/step - loss: 0.0381 - accuracy: 0.9891 - val_loss: 1.5953 - val_accuracy: 0.6482
Epoch 115/200
64/64 [==============================] - 25s 379ms/step - loss: 0.0682 - accuracy: 0.9767 - val_loss: 7.9638 - val_accuracy: 0.3676
Epoch 116/200
64/64 [==============================] - 25s 381ms/step - loss: 0.0341 - accuracy: 0.9876 - val_loss: 2.8797 - val_accuracy: 0.5613
Epoch 117/200
64/64 [==============================] - 24s 371ms/step - loss: 0.0600 - accuracy: 0.9797 - val_loss: 1.3891 - val_accuracy: 0.7115
Epoch 118/200
64/64 [==============================] - 24s 376ms/step - loss: 0.0204 - accuracy: 0.9951 - val_loss: 1.3743 - val_accuracy: 0.6957
Epoch 119/200
64/64 [==============================] - 24s 371ms/step - loss: 0.0849 - accuracy: 0.9762 - val_loss: 1.6932 - val_accuracy:

64/64 [==============================] - 23s 355ms/step - loss: 0.0129 - accuracy: 0.9946 - val_loss: 0.8591 - val_accuracy: 0.8142
Epoch 170/200
64/64 [==============================] - 27s 424ms/step - loss: 0.0224 - accuracy: 0.9906 - val_loss: 0.9267 - val_accuracy: 0.8300
Epoch 171/200
64/64 [==============================] - 24s 362ms/step - loss: 0.0133 - accuracy: 0.9960 - val_loss: 1.6443 - val_accuracy: 0.7391
Epoch 172/200
64/64 [==============================] - 24s 360ms/step - loss: 0.0175 - accuracy: 0.9936 - val_loss: 0.8937 - val_accuracy: 0.8498
Epoch 173/200
64/64 [==============================] - 22s 335ms/step - loss: 0.0138 - accuracy: 0.9960 - val_loss: 1.2944 - val_accuracy: 0.7510
Epoch 174/200
64/64 [==============================] - 21s 320ms/step - loss: 0.0770 - accuracy: 0.9787 - val_loss: 2.8621 - val_accuracy: 0.5771
Epoch 175/200
64/64 [==============================] - 22s 332ms/step - loss: 0.0480 - accuracy: 0.9842 - val_loss: 0.9096 - val_accuracy: